# ![icon](./images/uva-icon-57x57.png) WEEK 09 Docker

| Working Efficiently With Software |
| :--: |
| Data Engineering |
| School of Data Science |
| University of Virginia |

# ![icon](./images/uva-icon-57x57.png) Announcements & Agenda

---

**NB!: Please remember your PRs will 'BOUNCE'**
* **If the description is not filled out properly.**
* **If your branch name is not named properly.**

---


## T-Shirt sizing the remaining labs.
- ~~Setting up snowflake connector.  **M/L**~~
- Dockerizing our pipeline.  **XL** **<===== AWS ⬆️ == Snowflake ⬇️ ===**
- Git repo in snowflake.  **M**
- DBT-expectations **M**
- Snowflake CORTEX.  **S**


## Participation Reminder
* Class attendance
* Cameras on please

# ![icon](./images/uva-icon-57x57.png) 🏛️ From Chroot Jail to Docker Container

Before modern containerization, Linux systems used a primitive form of isolation called a **chroot jail**. Modern Docker containers are the direct evolutionary descendants of this concept, supercharged by modern kernel features.

---

### 1. The Primitive Ancestor: `chroot` (1979)
The `chroot` (change root) system call changes the root directory of the current running process and its children.

* **How it works:** It tricks a process into believing a specific subdirectory (e.g., `/home/jail`) is the actual system root (`/`).
* **The Illusion:** The process cannot see or access files outside of this directory tree.
* **The Flaw:** It only isolates the **filesystem**. A process inside a chroot jail still shares the host's network, process table (`pid`), user accounts, and memory space. If it gains `root` privileges, it can easily break out.

---

### 2. The Modern Evolution: Namespaces & Cgroups
Docker takes the core idea of a jail (restricting what a process can see) and uses modern Linux kernel features to seal the remaining security gaps.

```
┌────────────────────────────────────────────────────────┐
│                   DOCKER CONTAINER                     │
│                                                        │
│  🔒 Chroot/Pivot_root ──> Isolates the Filesystem       │
│  🔒 Namespaces        ──> Isolates View (PID, Net, UID) │
│  🔒 Cgroups           ──> Limits Resources (CPU, RAM)  │
└────────────────────────────────────────────────────────┘
```

* **Namespaces (Isolation):** Virtualizes system resources. A container gets its own isolated network stack (`net`), process tree (`pid`), mount points (`mnt`), and user tables (`user`). It feels like a completely separate OS.
* **Control Groups / cgroups (Throttling):** Restricts physical resource consumption. It prevents a "jailed" process from consuming 100% of the host's CPU or memory.

---

### 3. How Docker Bridges the Gap

> 💡 **The Takeaway:** A Docker Image is essentially a pre-packaged, frozen **chroot directory template**, and a running Docker Container is that directory activated inside a high-security Linux **Namespace + Cgroup jail**.

| Capability | Ancient `chroot` Jail | Modern Docker Container |
| :--- | :--- | :--- |
| **Filesystem Scope** | Isolated to a specific subfolder. | Isolated via `pivot_root` using stacked image layers. |
| **Process Isolation** | **None.** Can see and kill host processes. | **Complete.** Has its own isolated PID 1 process tree. |
| **Network Isolation** | **None.** Shares the host network interfaces. | **Complete.** Gets a virtualized network card and private IP. |
| **Resource Limits** | **None.** Can consume all host RAM/CPU. | **Strict.** Cgroups enforce exact memory/CPU hard limits. |

# The Docker Lifecycle Loop

A Docker project follows a continuous feedback loop: you define the template (**Dockerfile**), freeze it into a static package (**Image**), boot it up as an isolated execution environment (**Container**), debug/modify your code, and loop back.

Below is the complete lifecycle, including running states and the development iteration loop.

**NB: Some cross concepts**
* **An image is much like a python class, a container is much like a python 'object' or 'instance'**
* **Note the terminology in stop/start as it compares to your AWS instance**

```mermaid
graph TD
    %% 1. The Build Phase (Static Template)
    subgraph Build Phase [1. Define & Compile]
        DF[📄 Dockerfile] -- "docker build" --> IM((📦 Docker Image))
        style DF fill:#f9f,stroke:#333,stroke-width:2px
        style IM fill:#bbf,stroke:#333,stroke-width:2px
    end

    %% 2. The Execution Phase (Running Container)
    subgraph Run Phase [2. Execution States]
        IM -- "docker run" --> RUN[🟢 Running Container]
        RUN -- "docker stop (SIGTERM)" --> STP[🔴 Stopped Container]
        STP -- "docker start" --> RUN
        RUN -- "docker kill (SIGKILL) / Crash" --> STP
    end

    %% 3. The Development Loop (Feedback/Iteration)
    subgraph Dev Loop [3. Iteration Loop]
        RUN -- "Examine Logs / Errors" --> DBG{🔍 Debug & Code Changes}
        DBG -- "Update Code / Dockerfile" --> DF
        STP -- "docker rm" --> RM[🗑️ Cleaned Up]
    end

    %% Styling
    style RUN fill:#9f9,stroke:#333,stroke-width:2px
    style STP fill:#f99,stroke:#333,stroke-width:2px
    style DBG fill:#ff9,stroke:#333,stroke-width:1px
    style RM fill:#ddd,stroke:#333,stroke-width:1px
```

### Key Phases Explained:

1. **The Build Phase:** You write structural instructions in a `Dockerfile`. Running `docker build` compiles this file top-to-bottom into an immutable, multi-layered **Docker Image** (the "class" or "blueprint").
2. **The Run Phase:** Running `docker run` instantiates the image into a live, active **Container** (the "object"). 
   * **Running:** The container is actively executing its primary process (PID 1).
   * **Stopped:** The process has terminated, but the container's scratch space, file modifications, and logs still exist on disk. It can be restarted or deleted.
3. **The Iteration Loop:** In data science or software development, you'll rarely write a perfect build on the first try. You run the container, inspect outputs or logs, find bugs, modify your source files, and **rebuild the image** to generate a new, updated container instance.

# ![icon](./images/uva-icon-57x57.png)  The Core Dockerfile Instructions

A `Dockerfile` is a recipe where each instruction creates a new read-only layer in your image. Here are the 5 essential commands you need to build almost any data science application.

---

### The Essential 5

| Instruction | What It Does | Example | Why It Matters |
| :--- | :--- | :--- | :--- |
| **`FROM`** | Sets the foundation (Base Image). | `FROM python:3.11-slim` | Every Dockerfile *must* start with this. It pulls the pre-configured OS and runtime environment. |
| **`WORKDIR`** | Sets the active directory inside the container. | `WORKDIR /app` | Acts like a permanent `cd`. All subsequent commands (`RUN`, `COPY`, `CMD`) will execute inside this directory. |
| **`COPY`** | Copies files from host to container. | `COPY requirements.txt .` | Moves local code, configurations, or dataset files into the container's isolated file system. |
| **`RUN`** | Executes commands **during the build process**. | `RUN pip install -r requirements.txt` | Used to install packages, compile code, or set up libraries. These changes are baked permanently into the image layers. |
| **`CMD`** | Sets the **default execution command** on boot. | `CMD ["python", "main.py"]` | Does not run during the build. Instead, it defines the default process that launches when you finally trigger `docker run`. |

---

### Pro-Tip: Layer Caching Efficiency
Docker builds images top-to-bottom. If a layer hasn't changed, Docker reuses a cached version. 

To speed up your builds, **always copy your dependency manifests and install them before copying your actual codebase**:

```dockerfile
# 1. Base setup
FROM python:3.11-slim
WORKDIR /app

# 2. Install dependencies (These rarely change, so Docker caches this layer!)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 3. Copy application code (This changes often, so put it last)
COPY . .

# 4. Default run command
CMD ["python", "app.py"]
```

# ![icon](./images/uva-icon-57x57.png)  Stacked Read-Only Layers
* UnionFS (Union Filesystem): Overlays multiple read-only filesystem directories, presenting them as a single virtual view.
* Each step in a Dockerfile creates a distinct, immutable, cacheable layer.
* Container Layer: Spawning a container adds a thin, temporary, writeable layer on top of the image structure.
* Shared base layers allow multiple active containers to operate with minimal physical disk overhead.
# ![icon](./images/docker_images-49.png) 

# ![icon](./images/uva-icon-57x57.png)  Docker Layers

# The Parallel Universe: Docker vs. Git

Docker images and Git repositories share a fundamental architectural philosophy: **Content-Addressable Snapshots & Layering**. If you understand Git commits, you already understand Docker layers.

---

### Structural Head-to-Head

| Concept | Git Version Control | Docker Containerization |
| :--- | :--- | :--- |
| **The Unit of Work** | **Commit** (Snapshot of tracking code changes) | **Layer** (Snapshot of a root filesystem state) |
| **Storage Model** | **Append-Only Directed Acyclic Graph** | **Read-Only Stacked Layers** (Union File System) |
| **Unique Identifier** | **SHA-1 / SHA-256 Hash** of the commit object | **SHA-256 Hash** of the layer content |
| **Base State** | Initial commit (`git init`) | Base Image / Parent Layer (`FROM ubuntu`) |
| **Working Copy** | **Working Directory** (Where you edit code files) | **Container Layer** (Thin, ephemeral writeable top-layer) |
| **Efficiency Mechanism** | Pointer-based references (untracked files aren't duplicated) | **Copy-on-Write (CoW)** (Unchanged layers are cached & reused) |

---

### The Anatomy of a Change

> 💡 **The Core Realization:** A Docker container running an application is structurally identical to checking out a Git branch, modifying files locally, and discarding the changes without committing.

* **Git Commit:** Captures a delta of text or files relative to the parent commit.
* **Docker Layer:** Captures a delta of binaries, libraries, or files added/modified relative to the underlying parent layer.

# ![icon](./images/uva-icon-57x57.png)  If you know GIT, you can undestand Docker layers!


The architectural mapping between Docker's inheritance model and Git's branching strategy is nearly identical. Using a base image to build a new containerized app is conceptually the same as creating a feature branch from a stable production branch.

---

### The Workflow Equivalence

```
 Git:     [ main branch ] ──────────────> [ git checkout -b feature ] ──> [ git commit ]
                                                                               │
 Docker:  [ FROM python:3.11 ] ─────────> [ Base Image Layers ] ────────> [ RUN/COPY Layers ]
```

| Action / Concept | Git Version Control | Docker Containerization |
| :--- | :--- | :--- |
| **The Foundation** | The `main` or `trunk` branch. | The Base Image (e.g., `FROM ubuntu`, `FROM python`). |
| **Starting New Work** | `git checkout -b feature/my-app` | The `FROM` instruction in a `Dockerfile`. |
| **Adding New Changes** | Staging and committing code changes (`git commit`). | Executing build steps (`RUN pip install`, `COPY . .`). |
| **Divergent History** | Creating multiple feature branches off `main` simultaneously. | Creating multiple target images using the same base layer. |

---

### The Divergence: Sharing vs. Merging

* **Perfect Disk Efficiency:** If you spin up 5 different custom app images all derived from `FROM python:3.11`, Docker **never duplicates** the base Python layers on your disk. They all point back to the exact same immutable parent layer hashes—exactly like Git branches tracking the same historical root commit.
* **One-Way Evolution:** Unlike Git, where feature branches eventually get merged back into `main`, **Docker images only evolve forward**. You never merge a child image back into its parent; instead, your finished child image simply becomes the new "Base Image" (`FROM my-packaged-app:v1`) for the next phase of deployment.

# The File Deletion Trap (Git vs. Docker)

Both Git and Docker use **append-only history**. If you commit a heavy file in Git, deleting it in a later commit doesn't shrink your `.git` repository size. The exact same trap applies to Docker layers.

---

### The Trap: Sequential `RUN` Commands
This approach appears clean, but it actually **bloats** your production image. 

```dockerfile
# Layer 1: Adds 500MB of build tools (Bakes 500MB permanently into history)
RUN apt-get update && apt-get install -y gcc

# Layer 2: Compiles your application code
RUN gcc my_app.c -o my_app

# Layer 3: Deletes the compiler
RUN apt-get remove -y gcc
```
* **Why this fails:** Layer 3 merely writes a "whiteout" metadata flag saying *“hide these files from the running container.”* The 500MB compiler remains forever embedded in the immutable history of Layer 1.

---

###  The Solution: Single-Layer Chaining (`&&`)
To keep the image lightweight, you must install, compile, and clean up **within the exact same instruction**.

```dockerfile
# Built as a single, highly optimized filesystem layer
RUN apt-get update && apt-get install -y gcc \
    && gcc my_app.c -o my_app \
    && apt-get purge -y --auto-remove gcc \
    && rm -rf /var/lib/apt/lists/*
```
* **Why this works:** Because the file creation, compilation, and deletion occur before the step completes, Docker only commits the final state (the compiled binary) to the image layer. The compiler never touches your permanent image history.

# ![icon](./images/uva-icon-57x57.png) Dockerizing

## This week, we look at packaging our pipeline in a docker image.

### `venv` vs. `Docker`

| Feature / Aspect | Python Virtual Environment (`venv`) — Pros | Python Virtual Environment (`venv`) — Cons | Docker Containerization — Pros | Docker Containerization — Cons |
| --- | --- | --- | --- | --- |
| **System & OS Dependencies** | **None.** No OS overhead; utilizes the host OS directly. | **No system isolation.** Cannot bundle non-Python dependencies (e.g., CUDA, LAPACK, `gfortran`, Graphviz, or spatial libraries like GDAL). Requires manual manual install on every host. | **Complete system isolation.** Packages the entire OS user space, system libraries, C/C++ compilers, and GPU drivers (via NVIDIA Container Toolkit). | **Heavy resource footprint.** Storage overhead is high (images easily swell to multiple gigabytes with ML libraries). |
| **Reproducibility** | **Fast & Lightweight.** Guarantees matching Python-level packages (via `requirements.txt` or `pyproject.toml`) with near-zero disk overhead. | **"Works on my machine" fragility.** Python packages with native C/C++ extensions (like NumPy, PyStan, or Tokenizers) might compile differently or fail depending on the host OS/architecture (e.g., Apple Silicon vs. Linux x86). | **Bit-for-bit reproducibility.** The exact same image runs identically on local machines, AWS EC2, or Kubernetes, regardless of the host OS. | **Layer caching pitfalls.** Poorly written Dockerfiles can lead to stale builds, slow compilation times, and massive image bloat. |
| **Local Development & Editor Tooling** | **Seamless integration.** Instantly recognized by terminal environments, shell configurations, shell completions, and editors (like Vim/Neovim, LSP servers, or debuggers). | **Environment pollution.** Accidentally installing packages globally if the environment isn’t active; local clutter. | **Isolated execution.** Keeps the local host completely clean; easily spin up supporting services (Redis, Postgres) alongside the app via `docker-compose`. | **Tooling friction.** Attaching local terminal-based debuggers, LSPs (like Pyright), and Linters to a running container requires complex volume mounting or SSH workarounds. |
| **Hardware & GPU Access (CUDA / Apple Silicon)** | **Direct Access.** Accesses local hardware (GPUs, CoreML) natively with zero translation layer or driver configuration. | **Manual Driver Matching.** The developer is responsible for manually matching local PyTorch/TensorFlow pre-built wheels with the exact host CUDA/cuDNN versions. | **Consistent ML Runtimes.** You can pin specific CUDA runtimes in the base image (e.g., `nvidia/cuda`), ensuring ML code runs predictably without local driver mismatch. | **Access Complexity.** Requires installing and maintaining the NVIDIA Container Toolkit on the host; passing GPU flags (`--gpus all`) to run containerized workloads. |
| **Cold Start & Cloud Execution** | **Sub-millisecond startup.** Applications boot instantly because there is no container runtime startup overhead. | **Slow deployment scaling.** If deploying bare-metal, installing dependencies on a fresh instance at runtime during autoscaling is incredibly slow. | **Standardized scaling.** Easily orchestrates via Kubernetes, ECS, or serverless container runtimes. Supports fast scaling of pre-built images. | **Cold start delays.** Pulling large ML/Data Science images (often 2GB to 5GB+) to a new cloud node can introduce significant latency. |

---

### Which should you choose for Data Science?

#### Use `venv` when:

1. **Pure Python workflows:** Your project relies entirely on standard PyPI packages that distribute pre-built binary wheels (e.g., Pandas, Scikit-Learn, requests).
2. **Local exploration and prototyping:** You are actively writing code, heavily debugging from the command line, running quick scripts, or doing exploratory data analysis where spinning up containers adds unnecessary friction.
3. **Severe storage constraints:** You want to avoid maintaining tens of gigabytes of cached Docker image layers on your local drive.

#### Use `Docker` when:

1. **Complex native dependencies:** You are using packages that require compilation or complex system libraries (e.g., PyStan, PyMC, spatial analysis tools, or specific C++ bindings).
2. **Heavy GPU acceleration:** You need a guaranteed CUDA setup that matches your remote training cluster or production runtime exactly.
3. **Microservice architectures & Production:** Your app needs to run as a reliable API (e.g., FastAPI serving a model) deployed to production via AWS Lambda (using container images), AWS ECS, or Kubernetes.


# ![icon](./images/uva-icon-57x57.png) About the lab

## DO NOT ADD THE .env FILE TO THE IMAGE

* **This is just as bad as checking credentials into github, dockerhub is also a popular hackable destination**

## What you'll do in the lab... `-i --env-file .env`


---

## You can pass in your .env keys, and run just like you did on your local

```
cat ids.txt | docker run -i --env-file .env <image name> bash -c "python clean.py | python extract.py"
```

---



## You will need to edit permissions on your instance to allow docker to work without SUDO

```
newgrp docker
```

---


## About the grading - use makefile jobs for the rubric

# ![icon](./images/uva-icon-57x57.png) Docker Cheat Sheet: Essential Commands

A practical reference guide for managing Docker images and containers from the terminal.

---

### Working with Images

| Action | Command | Description |
| :--- | :--- | :--- |
| **Build Image** | `docker build -t <image_name>:<tag> .` | Builds an image from the `Dockerfile` in the current directory (`.`). |
| **List Images** | `docker images` or `docker image ls` | Displays all locally cached images, their IDs, and sizes. |
| **Remove Image** | `docker rmi <image_id_or_name>` | Deletes a local image (fails if a container is using it). |
| **Prune Images** | `docker image prune` | Cleans up dangling/unused build layers to free up disk space. |

---

### Running and Managing Containers

| Action | Command | Description |
| :--- | :--- | :--- |
| **Run Interactive** | `docker run -it --name <name> <image> /bin/bash` | Spins up a container, maps your terminal keyboard/display (`-it`), and drops you into a bash shell. |
| **Run Background** | `docker run -d -p 8080:80 --name <name> <image>` | Runs the container in detached mode (`-d`) and maps host port `8080` to container port `80`. |
| **List Active** | `docker ps` | Lists all currently *running* containers. |
| **List All** | `docker ps -a` | Lists *all* containers (running, stopped, exited, or crashed). |
| **Stop Safely** | `docker stop <container_id_or_name>` | Sends a `SIGTERM` signal to let the app shut down gracefully. |
| **Kill Instantly** | `docker kill <container_id_or_name>` | Sends a `SIGKILL` signal to forcefully terminate the container immediately. |
| **Remove Container** | `docker rm <container_id_or_name>` | Deletes a stopped container configuration from disk. |

---

### Debugging and Inspection

| Action | Command | Description |
| :--- | :--- | :--- |
| **View Logs** | `docker logs -f <container_id_or_name>` | Tail and stream stdout/stderr output from a running background container (`-f` to follow). |
| **Execute Inside** | `docker exec -it <container_id_or_name> /bin/bash` | "Jump into" an already running background container to explore or run commands. |
| **See Layers** | `docker history <image_name>` | Shows the build history, instructions, and size breakdown per layer. |
| **Inspect Metadata**| `docker inspect <container_or_image>` | Returns a massive JSON payload containing network configurations, environment variables, and volumes. |

---

### Lifecycle Pro-Tips

* **The Ultimate Cleanup Command:**
  If you are running low on disk space from running multiple iterations of your data science apps, run:
  ```bash
  docker system prune -a --volumes
  ```
  *Warning: This forcefully removes all stopped containers, all unused networks, all unused images, and all build caches.*
* **Auto-Remove on Exit:**
  When testing or running interactive script jobs, use the `--rm` flag so the container automatically deletes itself the moment it exits:
  ```bash
  docker run --rm -it python:3.11-slim bash
  ```

```bash
## INTERACTIVE JOBS
### Already running
docker exec -it efrainolivares/ds5111-pipeline:latest /bin/bash

### Start from scratch
docker run --rm -it efrainolivares/ds5111-pipeline:latest /bin/bash

### and using an .env file interactive
docker run --rm -it --env-file .env efrainolivares/ds5111-pipeline:latest /bin/bash


### see what containers are running
docker ps

```